# Demo 2: The Fourth-Quarter Quality Packet

**Project so far.** Demo 1 gave Dr. Alvarez her 2025 wait scorecard: five clinics miss the 30-minute target, and Bayview's Orthopedics waits nearly doubled after September.

**This phase's ask.** Dr. Sam Whitfield, chair of the Quality Committee, needs the fourth-quarter (October to December) packet by January 9, 2026. The committee uses it to choose which visits its nurses review and what the health system publishes on its website. The packet has three parts:

- A visit-volume table in the committee's template, which lists all nine clinics in a fixed order and must account for every visit.
- A list of long waits for chart review. The two review nurses can pull about 5,000 charts a quarter.
- A public satisfaction table that hides any group of 10 or fewer patients.

The source is the same 1.2-million-visit extract as Demo 1, plus the committee's clinic list.

Run the cells from top to bottom.

In [ ]:
# Installs the course's pandas, pyarrow (for Parquet), and dowhy (for the data generator) in Colab (uv sync already did locally); if Colab asks, restart and rerun from the top
%pip install -q --no-warn-conflicts pandas==3.0.5 pyarrow==25.0.0 dowhy==0.14

This supplied cell downloads `generate_data.py` when it is missing, runs `generate_data.build()` to write the tables into `data/` if they are not there yet (about 30 seconds), then loads the clinic list and the visits.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/christopherseaman/datasci_217/main/08/demo"
if not Path("generate_data.py").exists():
    urlretrieve(f"{REPO_RAW}/generate_data.py", "generate_data.py")
if not all(Path(f"data/{name}.parquet").exists() for name in ["clinics", "visits", "labs"]):
    import generate_data
    generate_data.build()

clinics = pd.read_parquet("data/clinics.parquet")
visits = pd.read_parquet("data/visits.parquet")
print(clinics.shape, visits.shape)

Expect `(9, 3) (1200000, 11)`.

## Core walkthrough

### 1. Pull the fourth quarter

`visit_month` is text such as `2025-10`, so text comparison keeps October to December.

In [ ]:
display(clinics)
q4 = visits[visits["visit_month"] >= "2025-10"].copy()
print(f"Q4 visits: {len(q4):,}")
print(f"No clinic recorded: {q4['clinic'].isna().sum():,}")

Expect nine clinics in the committee's reporting order, ending with Oceanview, which opens on 2026-01-12, then 315,608 Q4 visits, 786 of them with no clinic recorded.

### 2. The template lists nine clinics; the summary shows eight

Start with the default summary.

In [ ]:
default = q4.groupby("clinic")["visit_id"].count()
display(default)
print(f"Visits counted: {default.sum():,} of {len(q4):,}")

Expect eight clinics in alphabetical order, no Oceanview, and 314,822 of 315,608 visits counted: the 786 visits with no clinic vanished without a warning. The template needs a fixed order, an Oceanview row, and the missing visits.

Declare the template's clinics and order with `pd.Categorical`, then ask for every category (`observed=False`) and the missing key (`dropna=False`).

In [ ]:
q4["clinic"] = pd.Categorical(q4["clinic"], categories=clinics["clinic"], ordered=True)
volume = q4.groupby("clinic", observed=False, dropna=False).agg(
    visits=("visit_id", "size"),
    patients=("patient_id", "nunique"),
    median_wait=("wait_min", "median"),
)
display(volume)
print(f"Visits counted: {volume['visits'].sum():,} of {len(q4):,}")

Expect ten rows: the nine clinics in the template's order, Oceanview with 0 visits, 0 patients, and a `NaN` median (no waits to take a median of), then a `NaN` row with the 786 unrecorded visits. Now all 315,608 visits are counted.

Draw the packet's chart. A chart axis needs text labels, so the `NaN` key gets the label `No clinic` (`astype(object)` turns the categorical labels into plain values that `fillna` can replace), and `ax.invert_yaxis()` puts the first clinic at the top.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

labels = volume.index.astype(object).fillna("No clinic")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(labels, volume["visits"])
ax.invert_yaxis()
ax.set(title="Q4 2025 visits by clinic, committee order", xlabel="Visits")
plt.show()

Expect ten bars from Mission at the top to `No clinic` at the bottom, with an empty slot for Oceanview (0 visits) and a sliver for the 786 unrecorded visits. The default summary would have drawn eight bars and said nothing about the other two.

**What we tell Dr. Whitfield:** the table lists all nine clinics and adds up to the 315,608 Q4 visits. 786 visits (0.2%) have no clinic recorded; registration has the list to correct.

### 3. Which waits deserve a chart review?

A 45-minute wait is routine in Orthopedics and alarming in Behavioral Health, so the review compares each visit with the usual wait in its own clinic and department. That needs a group summary on every visit row.

_Intentional mistake:_ attach each department's mean wait as a column.

In [ ]:
dept_mean = q4.groupby("department")["wait_min"].mean()
wrong = q4.copy()
wrong["dept_mean"] = dept_mean
print("Missing values in the new column:", wrong["dept_mean"].isna().sum())

try:
    wrong["dept_mean"] = dept_mean.values
except ValueError as error:
    print("ValueError:", error)

Expect `315608` missing values with no error: the summary is labeled by department and the visits by row number, so no label matches. Then `ValueError: Length of values (7) does not match length of index (315608)`: seven department means do not fit 315,608 rows.

**The fix:** `transform` returns one value per visit, aligned to the visits.

In [ ]:
cell = q4.groupby(["clinic", "department"], observed=True)["wait_min"]
q4["cell_mean"] = cell.transform("mean")
q4["cell_sd"] = cell.transform("std")
q4["wait_z"] = (q4["wait_min"] - q4["cell_mean"]) / q4["cell_sd"]
q4["long_for_cell"] = q4["wait_z"] > 3
q4["over_60"] = q4["wait_min"] > 60
display(q4[["clinic", "department", "wait_min", "cell_mean", "wait_z"]].head().round(2))

Expect five visits, each with its own cell's mean wait and a z-score, the number of standard deviations above that mean. The first, an 18-minute Cardiology wait at Mission, has a z-score of -0.78, below its cell's mean of 27.38. A visit with no recorded wait or no clinic gets a `NaN` z-score. The index keeps the visits' row numbers from the full extract (884392 onward).

The committee's old rule flagged every wait over 60 minutes. Compare it with a z-score rule (more than 3 standard deviations above the cell's mean), department by department: raw waits against the 60-minute line, and z-scores against the line at 3. `ax.axvline(x)` draws a vertical reference line at `x`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.boxplot(data=q4, x="wait_min", y="department", fliersize=1, ax=axes[0])
axes[0].axvline(60, color="red", linestyle="--")
axes[0].set(title="Raw wait: flat rule at 60 minutes", xlabel="Wait (minutes)", ylabel="")
sns.boxplot(data=q4, x="wait_z", y="department", fliersize=1, ax=axes[1])
axes[1].axvline(3, color="red", linestyle="--")
axes[1].set(title="Wait z-score within clinic and department: rule at 3", xlabel="z-score")
plt.show()

Expect the left boxes at very different positions: Orthopedics far right, Behavioral Health far left with no dot past the red line. Most dots past 60 belong to Orthopedics and Family Medicine. On the right every box is centered near 0 with a similar width, and every department, Behavioral Health included, has a tail of dots past 3.

In [ ]:
flags = q4.groupby("department").agg(
    visits=("visit_id", "size"),
    over_60=("over_60", "sum"),
    long_for_cell=("long_for_cell", "sum"),
)
display(flags)

Expect the flat rule to flag 5,336 visits, 4,908 of them in Orthopedics (3,414) and Family Medicine (1,494), and none in Behavioral Health. That is more than the nurses' 5,000 charts, and most of it is Orthopedics doing what Orthopedics usually does. The z-score rule flags 4,199 visits, within the budget, spread across every department, including 398 in Behavioral Health, where a 20-minute wait is far from usual.

Before recommending the z-score rule, check what it cannot see. A cell's own mean can be the problem: Demo 1 found Bayview's Orthopedics waits nearly doubled after September.

In [ ]:
bayview = q4[(q4["clinic"] == "Bayview") & (q4["department"] == "Orthopedics")]
print(f"Visits: {len(bayview):,}  mean wait: {bayview['wait_min'].mean():.1f}")
print("Over 60 minutes:", bayview["over_60"].sum(), " long for its cell:", bayview["long_for_cell"].sum())

Expect 6,191 visits with a mean wait of 60.7 minutes: 2,043 are over 60, but only 106 are long for their cell. Compared with its own slow quarter, almost nothing at Bayview looks unusual. The z-score finds odd visits within a group; the group summary (Demo 1's grid) finds the odd group.

**What we tell Dr. Whitfield:** switch the chart review to the z-score rule (4,199 charts, every department). Keep Bayview Orthopedics on the agenda as a staffing problem, because a chart review will not flag it.

### 4. Can every satisfaction cell be published?

The website shows mean satisfaction by clinic and department from Q4 survey respondents. The publication rule hides any group with 10 or fewer patients. First find the cells at risk.

In [ ]:
respondents = q4.dropna(subset=["satisfaction"])
cells = respondents.groupby(["clinic", "department"], observed=True).agg(
    surveys=("visit_id", "size"),
    patients=("patient_id", "nunique"),
)
display(cells[cells["patients"] < 11])

Expect one cell, Ridgecrest Cardiology: 23 surveys from only 10 patients, because its outreach clinic sees the same regulars every month and collects a survey at each visit.

A size rule is the obvious draft, but it counts surveys. `filter` keeps every row of the groups that pass a test, so the test decides what is published: counting rows would publish that cell; counting distinct patients hides it.

In [ ]:
by_cell = respondents.groupby(["clinic", "department"], observed=True)
rows_rule = by_cell.filter(lambda g: len(g) >= 11)
patients_rule = by_cell.filter(lambda g: g["patient_id"].nunique() >= 11)
print(f"Respondents: {len(respondents):,}")
print(f"Kept by len(g) >= 11:     {len(rows_rule):,}")
print(f"Kept by nunique() >= 11:  {len(patients_rule):,}")

public = pd.pivot_table(patients_rule, values="satisfaction", index="department",
                        columns="clinic", aggfunc="mean", observed=True)
display(public.round(2))

Expect 110,760 respondents. The row rule keeps 110,472, every respondent with a clinic (the 288 with no clinic have no group), and the patient rule keeps 23 fewer, 110,449. In the public table, Ridgecrest's Cardiology cell is `NaN` (suppressed), beside the cells for departments Ridgecrest does not run.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(public, annot=True, fmt=".2f", cmap="RdYlGn", ax=ax)
ax.text(list(public.columns).index("Ridgecrest") + 0.5, list(public.index).index("Cardiology") + 0.5,
        "suppressed", ha="center", va="center", fontsize=8)
ax.set(title="Mean satisfaction (1 to 5), Q4 2025, groups of 11+ patients", xlabel="Clinic", ylabel="")
plt.show()

Expect Ridgecrest's Cardiology cell labeled `suppressed` and its three other blank cells, the departments it does not run, left empty. Scores follow waits: the Behavioral Health row, the shortest wait, is the greenest at every clinic but Ridgecrest, where Pediatrics edges it (4.37 against 4.26), and Bayview's Orthopedics, the longest wait, is the reddest cell (2.59).

**What we tell Dr. Whitfield:** the table is ready to publish with one cell suppressed, Ridgecrest Cardiology (10 patients).

### 5. Which two visits should each clinic's reviewer pull first?

The chart reviewers start with each clinic's two longest waits, as whole rows with department and date. `groupby(...).apply(func)` runs a function on each clinic's rows and stitches the results together; `g.nlargest(2, "wait_min")` returns the two rows of group `g` with the longest waits. `include_groups=False` leaves the grouping column out of `g`.

In [ ]:
top2 = q4.groupby("clinic", observed=True).apply(
    lambda g: g.nlargest(2, "wait_min"), include_groups=False
)
display(top2[["department", "visit_month", "wait_min"]].head(4))
print(top2.shape, top2.index.nlevels)

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=top2.reset_index(level=0), x="wait_min", y="clinic", hue="department", s=80, ax=ax)
ax.set(title="Two longest Q4 waits per clinic", xlabel="Wait (minutes)", ylabel="")
plt.show()

Expect `(16, 15) 2`: 2 visits for each of the 8 clinics with visits (Oceanview and the unrecorded clinic have none to show), with the columns of `q4` so far. The index has two levels, the clinic in committee order and the original row number. The first four rows are Mission's Family Medicine waits of 240 and 191 minutes, then Sunset's Orthopedics waits of 170 and 158. The chart shows one pair of dots per clinic, from Ridgecrest's 98 minutes, the lowest of the 16, up to the 240-minute maximum.

The same rows without `apply`: sort the waits, then keep the first two rows of each group.

In [ ]:
flat = q4.sort_values("wait_min", ascending=False, kind="stable").groupby("clinic", observed=True).head(2)
print(flat.shape, flat.index.nlevels)
print("Same visits:", set(flat["visit_id"]) == set(top2["visit_id"]))

Expect `(16, 16) 1` and `Same visits: True`: the same 16 visits with a flat row-number index and the clinic still a column. `apply` fits when the per-group step is more than one built-in call; the sort-and-`head` version is faster on 315,608 rows.

**What we tell Dr. Whitfield:** the reviewers start with 16 visits, two per clinic. Four are at 240 minutes (two at Bayview, one each at Excelsior and Mission), the highest wait recorded, so the true waits may be longer.

## Independent practice

Run the core walkthrough first. Each item is a request that came back from the committee.

### 6. The committee asks what Excelsior's October waits would have been

Excelsior's kiosk was down for most of October. Fill each missing wait with the median of its own clinic and department, a group-specific fill with `transform`, and compare it with one overall median.

In [ ]:
october = q4[q4["visit_month"] == "2025-10"]
display(october.groupby("clinic", observed=True)["wait_min"].agg(["size", "count"]))

cell_median = q4.groupby(["clinic", "department"], observed=True)["wait_min"].transform("median")
q4["wait_filled"] = q4["wait_min"].fillna(cell_median)
overall_fill = q4["wait_min"].fillna(q4["wait_min"].median())
excelsior_ortho = (q4["clinic"] == "Excelsior") & (q4["department"] == "Orthopedics") & q4["wait_min"].isna()
print("Excelsior Orthopedics, cell-median fill:", q4.loc[excelsior_ortho, "wait_filled"].unique())
print("Excelsior Orthopedics, overall fill:   ", overall_fill[excelsior_ortho].unique())

Expect Excelsior's October `count` (5,309) far below its `size` (13,190): its kiosk was down for most of the month. The cell-median fill gives Excelsior's missing Orthopedics waits `[29.]`, its own Orthopedics median; the overall fill gives them `[17.]`, the all-visit median. Waits with no clinic recorded stay missing, because their group has no key.

### 7. Finance wants the volume table as a clinic-by-department grid

The same coverage choices, `observed=False` and `dropna=False`, work in `pivot_table`.

In [ ]:
volume_grid = pd.pivot_table(q4, values="visit_id", index="clinic", columns="department",
                             aggfunc="count", observed=False, dropna=False, fill_value=0)
display(volume_grid)
print("Rows:", len(volume_grid), " total:", volume_grid.to_numpy().sum())

Expect 10 rows: a `NaN` row for the unrecorded visits (listed first here, so find it by its label), the nine clinics, and Oceanview all zeros. The total is 315,608. A count of 0 is a real value here, so `fill_value=0` is right.

### 8. The staffing subcommittee asks where patient time goes

Two `filter` rules on the clinic and department cells: a mean wait over 40 minutes (slow cells) and a total over 365,000 minutes (cells where patients spend the most time).

In [ ]:
by_cell_all = q4.groupby(["clinic", "department"], observed=True)
slow = by_cell_all.filter(lambda g: g["wait_min"].mean() > 40)
heavy = by_cell_all.filter(lambda g: g["wait_min"].sum() > 365_000)
print("Mean wait over 40 minutes:", slow.groupby(["clinic", "department"], observed=True).size().to_dict())
display(heavy.groupby(["clinic", "department"], observed=True)["wait_min"].agg(["size", "sum", "mean"]))

Expect one cell by mean, Bayview Orthopedics with 6,191 visits. By total minutes, one cell: Family Medicine at Mission (585,878 minutes), with a mean wait of about 25: not slow, but where the most patient time goes. Bayview Orthopedics records 328,780 minutes, below the line, but the kiosk lost 777 of its 6,191 waits, many of them long (Demo 0); at its own mean of 60.7, those would add about 47,000 minutes and put it over.

A row mask with the same threshold answers a different question: it keeps single long waits from every cell.

In [ ]:
long_rows = q4[q4["wait_min"] > 40]
print(f"Row mask: {len(long_rows):,} visits from {len(long_rows.groupby(['clinic', 'department'], observed=True).size())} cells")
print(f"filter:   {len(slow):,} visits from 1 cell")

Expect the row mask to keep 23,762 visits from 40 cells, and `filter` all 6,191 visits, short waits included, from Bayview Orthopedics.

### 9. A reviewer rebuilds the z-scores with a lambda

A reviewer on the committee recomputes the z-scores with `transform` and a `lambda` to check step 3.

In [ ]:
z_lambda = cell.transform(lambda x: (x - x.mean()) / x.std())
print("Same index as q4?", z_lambda.index.equals(q4.index))
print("Largest difference:", (z_lambda - q4["wait_z"]).abs().max())

Expect `True` and a difference near `4e-14`: the same z-scores, so the packet's flags stand.

## Optional extension: the visits a pivot table of means drops

Prerequisite: BONUS.md, Group Coverage Edge Cases.

Dr. Whitfield asks whether the 786 unrecorded visits waited longer than the rest. A pivot table's `dropna` setting decides whether they appear at all.

In [ ]:
for drop in [True, False]:
    means = pd.pivot_table(q4, values="wait_min", index="clinic", columns="department",
                           aggfunc="mean", observed=False, dropna=drop)
    print(f"dropna={drop}: {len(means)} rows")
display(means.round(1))

Expect 8 rows with `dropna=True`: the `NaN` group goes because its key is missing, and Oceanview goes because every one of its cells is `NaN`. With `dropna=False`, 10 rows. The `NaN` row, listed first, holds real means, such as 39.5 minutes in Orthopedics, computed from the 786 visits the default report never mentions. They sit within the range of the clinics' own means, so the unrecorded visits look like a registration slip, not a different kind of visit.